In [6]:
import numpy as np
import pandas as pd
import re

In [1]:
from sklearn.neural_network import MLPRegressor
from src import calculate_metrics

In [ ]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("MLP_Regressor_HPO")
mlflow.set_tracking_uri("http://localhost:5000")

In [2]:
import json
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, HalvingRandomSearchCV

In [3]:
param_list = {}

## Preparacion de datos

In [ ]:
df_train = pd.read_csv('../data/train/transformed_train.csv')
df_val = pd.read_csv('../data/train/transformed_val.csv')

X_train = df_train.drop(columns=["Precio"])
y_train = df_train["Precio"]

X_val = df_val.drop(columns=["Precio"])
y_val = df_val["Precio"]

## First Experiment

Ajuste de hiperparametros

In [20]:
param_dist = {
    'hidden_layer_sizes': [
        (150, 100), (200, 150, 100), (250, 200, 150),
        (300, 200, 100), (300, 250, 200, 100)
    ],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],  
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.0001, 0.0005, 0.001, 0.005],
    'max_iter': [1000, 1500],
    'early_stopping': [True],
    'validation_fraction': [0.1, 0.15],
    'n_iter_no_change': [10, 15, 20],
    'batch_size': [32, 64, 128, 'auto']
}

mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, np.log1p(y_train)) 
best_model = search.best_estimator_
best_params = search.best_params_

y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   5.18e+14   |   4.25e+14
R²           |   0.2528     |   0.3178
MAE          |   1.16e+07   |   1.10e+07
EVS          |   0.2761     |   0.3363
Max Error    |   7.34e+08   |   3.93e+08
Median AE    |   6.97e+06   |   6.60e+06

Best parameters found:
{
    "validation_fraction": 0.15,
    "solver": "adam",
    "n_iter_no_change": 20,
    "max_iter": 1500,
    "learning_rate_init": 0.0005,
    "learning_rate": "constant",
    "hidden_layer_sizes": [
        150,
        100
    ],
    "early_stopping": true,
    "batch_size": 128,
    "alpha": 0.01,
    "activation": "tanh"
}


In [ ]:
y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   5.88e+14   |   5.03e+14
R²           |   0.1505     |   0.1929
MAE          |   1.28e+07   |   1.23e+07
EVS          |   0.1694     |   0.2082
Max Error    |   7.47e+08   |   4.02e+08
Median AE    |   8.28e+06   |   7.97e+06

Best parameters found:
{
    "solver": "adam",
    "max_iter": 1000,
    "learning_rate_init": 0.005,
    "learning_rate": "constant",
    "hidden_layer_sizes": [
        50,
        50
    ],
    "alpha": 0.001,
    "activation": "tanh"
}


In [ ]:
param_dist = {
    'hidden_layer_sizes': [
        (100, 100),
        (150, 100),
        (200, 100),
        (100, 100, 50),
        (200, 150, 100),
        (300, 200, 100)
    ],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam'],
    'alpha': [1e-4, 1e-3, 1e-2, 0.05],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.0005, 0.001, 0.005, 0.01],
    'max_iter': [1000, 1500],
    'early_stopping': [True],
    'validation_fraction': [0.1],
    'n_iter_no_change': [10, 15],
    'batch_size': ['auto', 64, 128]
}

mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

search = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, np.log1p(y_train))  
best_model = search.best_estimator_
best_params = search.best_params_

y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)

param_list["RandomizedSearchCV"] = best_params

print("Best parameters found:")
print(json.dumps(best_params, indent=4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   5.07e+14   |   4.28e+14
R²           |   0.2681     |   0.3134
MAE          |   1.12e+07   |   1.08e+07
EVS          |   0.2872     |   0.3298
Max Error    |   7.26e+08   |   3.95e+08
Median AE    |   6.62e+06   |   6.44e+06

Best parameters found:
{
    "validation_fraction": 0.1,
    "solver": "adam",
    "n_iter_no_change": 15,
    "max_iter": 1000,
    "learning_rate_init": 0.0005,
    "learning_rate": "constant",
    "hidden_layer_sizes": [
        200,
        150,
        100
    ],
    "early_stopping": true,
    "batch_size": "auto",
    "alpha": 0.0001,
    "activation": "tanh"
}


In [16]:
param_dist = {
    'hidden_layer_sizes': [(100,100), (100, 200), (200, 200), (500, 100)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [1e-3, 1e-2],
    'learning_rate': ['constant', 'adaptive'],
    'learning_rate_init': [0.001, 0.005],
    'max_iter': [500, 1000]
}

mlp = MLPRegressor(
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

halving_search = HalvingRandomSearchCV(
    estimator=mlp,
    param_distributions=param_dist,
    factor=3,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

halving_search.fit(X_train, np.log1p(y_train))

best_model = halving_search.best_estimator_
best_params = halving_search.best_params_

y_pred_train = np.expm1(best_model.predict(X_train))
y_pred_val = np.expm1(best_model.predict(X_val))

calculate_metrics(y_train, y_pred_train, y_val, y_pred_val)
param_list["HalvingRandomSearchCV_MLP"] = best_params

print("Best parameters found:")
print(best_params)

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 7
min_resources_: 6
max_resources_: 11682
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 128
n_resources: 6
Fitting 3 folds for each of 128 candidates, totalling 384 fits


c:\Users\ferna\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 128 is smaller than n_iter=1947. Running 128 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


----------
iter: 1
n_candidates: 43
n_resources: 18
Fitting 3 folds for each of 43 candidates, totalling 129 fits
----------
iter: 2
n_candidates: 15
n_resources: 54
Fitting 3 folds for each of 15 candidates, totalling 45 fits
----------
iter: 3
n_candidates: 5
n_resources: 162
Fitting 3 folds for each of 5 candidates, totalling 15 fits
----------
iter: 4
n_candidates: 2
n_resources: 486
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   7.17e+14   |   6.45e+14
R²           |   -0.0350     |   -0.0348
MAE          |   1.42e+07   |   1.38e+07
EVS          |   0.0156     |   0.0094
Max Error    |   7.66e+08   |   4.20e+08
Median AE    |   9.31e+06   |   8.94e+06

Best parameters found:
{'solver': 'adam', 'max_iter': 1000, 'learning_rate_init': 0.001, 'learning_rate': 'constant', 'hidden_layer_sizes': (100, 100), 'alpha': 0.01, 'activation': 'tanh'}


In [ ]:
input_example = X_train.iloc[0:1]

for name, params in param_list.items():
    if 'early_stopping' in params:
        params['early_stopping'] = True
    if 'validation_fraction' in params:
        params['validation_fraction'] = 0.1
    if 'n_iter_no_change' in params:
        params['n_iter_no_change'] = 10
    if 'random_state' in params:
        params['random_state'] = 42
    
    model = MLPRegressor(**params)
    model.fit(X_train, np.log1p(y_train))

    y_pred_train = np.expm1(model.predict(X_train))
    y_pred_val = np.expm1(model.predict(X_val))

    metrics = calculate_metrics(y_val, y_pred_val, verbose=False)

    run_name = f"MLP_{name}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        signature = infer_signature(input_example, model.predict(input_example))
        mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example, signature=signature)
